<a href="https://colab.research.google.com/github/walidsafaa/OSU/blob/main/Torrent_To_Google_Drive_Downloader_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader v2

### Mount Google Drive
To stream files we need to mount Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###Dependency
https://www.libtorrent.org/

In [2]:
!apt install python3-libtorrent

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3-libtorrent is already the newest version (2.0.5-5).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


### Code to download torrent
Variable **link** stores the link string.

In [15]:
# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & UPLOAD TORRENT FILE
# ==============================================================================
from google.colab import drive, files
import os
import shutil

drive.mount('/content/drive', force_remount=False)

# Local temporary path (avoids memory-mapping filesystem errors)
temp_local_path = '/content/downloads/'
# Final Google Drive target path
final_drive_path = '/content/drive/My Drive/Torrent/'

os.makedirs(temp_local_path, exist_ok=True)
os.makedirs(final_drive_path, exist_ok=True)

print("Please select your .torrent file to upload:")
uploaded = files.upload()

torrent_file_path = list(uploaded.keys())[0]
print(f"\nLoaded file: {torrent_file_path}")

# ==============================================================================
# 2. PRIVATE TORRENT CONFIGURATION
# ==============================================================================
import libtorrent as lt
import time
import datetime

settings = {
    'user_agent': 'qBittorrent/4.6.3',
    'enable_dht': False,                     # Private tracker requirement
    'enable_lsd': False,
    'listen_interfaces': '0.0.0.0:6881',
    'active_downloads': 5,
    'download_rate_limit': 0,
    'upload_rate_limit': 0
}

ses = lt.session(settings)

# Parse torrent info
info = lt.torrent_info(torrent_file_path)

atp = lt.add_torrent_params()
atp.ti = info
# Save to LOCAL filesystem first to prevent "No such device" mmap error
atp.save_path = temp_local_path

handle = ses.add_torrent(atp)

begin = time.time()
print("Start Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Downloading: {handle.status().name}")

state_str = ['queued', 'checking', 'downloading metadata',
             'downloading', 'finished', 'seeding', 'allocating']

# ==============================================================================
# 3. DOWNLOAD LOOP
# ==============================================================================
print("\nConnecting to Private Tracker & Peers...\n")

time.sleep(2)

while handle.status().state != lt.torrent_status.seeding:
    s = handle.status()

    # Filter real system errors
    if s.errc and s.errc.value() != 0:
        print(f"\n[Tracker Error #{s.errc.value()}]: {s.errc.message()}")
        break

    print(
        f"\r{s.progress * 100:.2f}% | "
        f"Down: {s.download_rate / 1000:.1f} kB/s | "
        f"Up: {s.upload_rate / 1000:.1f} kB/s | "
        f"Peers: {s.num_peers} | "
        f"Status: {state_str[s.state]}",
        end=""
    )
    time.sleep(3)

end = time.time()
torrent_name = handle.status().name

print(f"\n\nDownload complete locally! Moving {torrent_name} to Google Drive...")

# ==============================================================================
# 4. MOVE COMPLETED FILE TO GOOGLE DRIVE
# ==============================================================================
local_file_location = os.path.join(temp_local_path, torrent_name)
drive_file_location = os.path.join(final_drive_path, torrent_name)

if os.path.exists(local_file_location):
    shutil.move(local_file_location, drive_file_location)
    print(f"Successfully moved to: {drive_file_location}")
else:
    print(f"Transferred to Google Drive output folder.")

print(f"Total Elapsed Time: {int((end-begin)//60)} min : {int((end-begin)%60)} sec")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Please select your .torrent file to upload:


Saving [www.arabp2p.net]_-_زوتروبوليس 2 [جـ2] [H264] [1080p] [DSNP] Zootopia 2.torrent to [www.arabp2p.net]_-_زوتروبوليس 2 [جـ2] [H264] [1080p] [DSNP] Zootopia 2 (3).torrent

Loaded file: [www.arabp2p.net]_-_زوتروبوليس 2 [جـ2] [H264] [1080p] [DSNP] Zootopia 2 (3).torrent
Start Time: 2026-07-28 15:05:19
Downloading: Zootropolis 2 2025 1080p DSNP WEB-DL DDP5 1 H 264.TEEFA.mkv

Connecting to Private Tracker & Peers...

99.73% | Down: 235.2 kB/s | Up: 7.2 kB/s | Peers: 4 | Status: downloading

Download complete locally! Moving Zootropolis 2 2025 1080p DSNP WEB-DL DDP5 1 H 264.TEEFA.mkv to Google Drive...
Successfully moved to: /content/drive/My Drive/Torrent/Zootropolis 2 2025 1080p DSNP WEB-DL DDP5 1 H 264.TEEFA.mkv
Total Elapsed Time: 2 min : 47 sec
